<a href="https://colab.research.google.com/github/prateek-sahu/Discovery-of-Handwashing/blob/master/Autogen_RAG_Tutorial_for_ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install autogen==0.8.6
import autogen
import os

os.environ["AUTOGEN_USE_DOCKER"] = "False"

llm_config = {
    "config_list": [{"model": "gpt-4o", "api_key": "your_api_key"}],
}

from autogen import AssistantAgent, UserProxyAgent

assistant=AssistantAgent("assistant", llm_config=llm_config)
user_proxy = UserProxyAgent("user_proxy", human_input_mode="NEVER",llm_config=llm_config)

user_proxy.initiate_chat(assistant, message="Hello!  How are you today?", max_turns=1)
!pip install chromadb
import chromadb

chroma_client = chromadb.Client()

products_collection = chroma_client.create_collection(name="products")
orders_collection = chroma_client.create_collection(name="orders")

products_collection.add(
    documents=["Frisbee", "Dog Bowl"],
    ids=["id1", "id2"],
)

orders_collection.add(documents=["1234", "5678"], ids=["id1", "id2"])

import csv

with open("products.csv") as file:
    product_lines = csv.reader(file)

    id = 1

    for line in product_lines:
        if line[0] == "Product Name":
            continue
        else:
            products_collection.add(documents=str(line), ids=str(id))
            id += 1

with open("orders.csv") as file:
    order_lines = csv.reader(file)

    id = 1

    for line in order_lines:
        if line[0] == "Order Number":
            continue
        else:
            orders_collection.add(documents=str(line), ids=str(id))
            id += 1


def search_products(search_query):
    results = products_collection.query(query_texts=[search_query], n_results=2)
    return results


def search_orders(search_query):
    results = orders_collection.query(query_texts=[search_query], n_results=2)
    return results


products_search_results = search_products("What is Artisanal Air?")
print(products_search_results["documents"])

orders_search_results = search_orders("Earl Eclair")
print(orders_search_results["documents"])
import asyncio
from typing_extensions import Annotated

config_list = [{"model": "gpt-4o", "api_key": "your api key here"}]

products_search_assistant_agent = autogen.AssistantAgent(
        name="products_search_assistant",
        system_message="""You are a helpful assistant.
You have access to a database of documents about products, and you may search it.
The correct syntax for a search is: "what you want to search for".
Please use the search function to find information which can be used to answer the user's question.
DO NOT rely on your own knowledge, ONLY use the information retrieved from the search.
Please be honest about what you find. I am not looking for perfection, just the truth.
You are amazing and you can do this. I will pay you $200 for an excellent result,
but only if you follow all instructions exactly.""",
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
        code_execution_config=False,
    )

orders_search_assistant_agent = autogen.AssistantAgent(
        name="orders_search_assistant",
        system_message="""You are a helpful assistant.
You have access to a database of documents about customer orders,
and you may search it. The correct syntax for a search is: "what you want to search for".
Please use the search function to find information which can be used to answer the user's question.
DO NOT rely on your own knowledge, ONLY use the information retrieved from the search.
Please be honest about what you find. I am not looking for perfection, just the truth.
You are amazing and you can do this. I will pay you $200 for an excellent result,
but only if you follow all instructions exactly.""",
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
        code_execution_config=False,
    )

products_search_executor_agent = autogen.UserProxyAgent(
        name="products_search_executor",
        code_execution_config=False,
        system_message="""When enough information has been retrieved to answer
          the user's question to full satisfaction,
          please return "TERMINATE" to end the conversation.
        If more information must be collected, please return CONTINUE.""",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=4,
        is_termination_msg=lambda x: x.get("content", "")
        .rstrip()
        .endswith("TERMINATE"),
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
    )

orders_search_executor_agent = autogen.UserProxyAgent(
        name="orders_search_executor",
        code_execution_config=False,
        system_message="""When enough information has been retrieved to answer
          the user's question to full satisfaction,
          please return "TERMINATE" to end the conversation.
        If more information must be collected, please return CONTINUE.""",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=4,
        is_termination_msg=lambda x: x.get("content", "")
        .rstrip()
        .endswith("TERMINATE"),
        llm_config={
            "config_list": config_list,
            "temperature": 0.0
        },
    )

products_groupchat = autogen.GroupChat(
        agents=[products_search_assistant_agent,
                products_search_executor_agent],
        messages=[],
        max_round=4,
        speaker_selection_method="round_robin",
    )

products_groupchat_manager = autogen.GroupChatManager(
        groupchat=products_groupchat,
        name="products_groupchat_manager",
        llm_config={"config_list": config_list},
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )

orders_groupchat = autogen.GroupChat(
        agents=[orders_search_assistant_agent,
                orders_search_executor_agent],
        messages=[],
        max_round=4,
        speaker_selection_method="round_robin",
    )

orders_groupchat_manager = autogen.GroupChatManager(
        groupchat=orders_groupchat,
        name="orders_groupchat_manager",
        llm_config={"config_list": config_list},
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )

async_chat_plan = [
        {
            "chat_id": 1,
            "recipient": products_groupchat_manager,
            "message": "What is Artisanal Air?.",
            "summary_method": "reflection_with_llm",
            "silent": False,
        },
        {
            "chat_id": 2,
            "recipient": orders_groupchat_manager,
            "message": "What did Ned Noodle order?",
            "summary_method": "reflection_with_llm",
            "silent": False,

        }
    ]


async def start_groupchat():
    user = autogen.UserProxyAgent(
        name="User",
        human_input_mode="NEVER",
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )
    await user.a_initiate_chats(async_chat_plan)

await start_groupchat()
# asyncio.run(start_groupchat())
@products_search_assistant_agent.register_for_execution()
@products_search_executor_agent.register_for_llm(description="Search a ChromaDB collection containing information about products.")
async def search_products(search_query: Annotated[str, "Search query"]):
    await asyncio.sleep(10)
    results = products_collection.query(query_texts=[search_query], n_results=2)
    return results

@orders_search_assistant_agent.register_for_execution()
@orders_search_assistant_agent.register_for_llm(description="Search a ChromaDB collection containing information about products.")
async def search_orders(search_query: Annotated[str, "Search query"]):
    await asyncio.sleep(10)
    results = orders_collection.query(query_texts=[search_query], n_results=2)
    return results

await start_groupchat()
#asyncio.run(start_groupchat())
writer_assistant_agent = autogen.AssistantAgent(
        name="writer_assistant",
        system_message="""You are a helpful assistant for a company.
Your job is to answer the user's question using the provided information.
DO NOT rely on your own knowledge, ONLY use the provided info.
If you don't know the answer, just say you don't know.
You are amazing and you can do this. I will pay you $200 for an excellent result, but only if you follow all instructions exactly.""",
        llm_config={
            "config_list": config_list,
            "temperature": 0,
        },
        code_execution_config=False,
        max_consecutive_auto_reply=1,
    )

retrieved_product_data = products_search_assistant_agent.chat_messages
retrieved_order_data = orders_search_assistant_agent.chat_messages

retrieved_data = "Retrieved product data: " + str(retrieved_product_data) + "  Retrieved order data: " + str(retrieved_order_data)
print(retrieved_data)
user_question = "What is Artisanal Air?"

writer_prompt = f"""Please write the final answer to the user's question: \n{user_question}\n\n
     The information retrieved from the search agents is:
     {retrieved_data}.  I will tip you $200 for an excellent result."""


writer_userproxy = autogen.UserProxyAgent(
        name="WriterUserproxy",  # no spaces in the agent name
        human_input_mode="NEVER",
        is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    )

writer_userproxy.initiate_chat(
        recipient=writer_assistant_agent, message=writer_prompt, silent=True
    )

result = writer_assistant_agent.chat_messages
print("FINAL RESULT:")
cleaned_result = list(result.items())[0][1][-2]["content"]
print(cleaned_result)
#More questions to ask:

#What is Stain Magnet T-Shirt?
#What is Disappointment Alarm Clock?
#Fake Plant That Collects Real Dust

#What did Admiral Bananaface order?
#Did Sergeant Noodles order a non-stick pan?
#What did Waffles McSprinkles order?
while True:
  answer = rag(user_question)
  if answer is True:
    break
  else:
    fix(function) OR fix (prompt)
